## Question 3: Exploring Anonymized Data - Can You Find the Golden Feature?



**What is Anonymized Data?**



Anonymized data is information that has been processed to remove or obscure personally identifiable details, making it impossible to trace back to specific individuals. In credit risk modeling, features are often anonymized (e.g., `P_2`, `D_39`, `B_1`) to protect customer privacy while still enabling powerful predictive analytics.



**The Challenge:**



Credit default prediction is central to managing risk in consumer lending. American Express, the world's largest payment card issuer, uses machine learning to predict whether cardholders will default on their payments. This helps optimize lending decisions and creates a better customer experience.



You are provided with an anonymized dataset containing behavioral and profile features. Your objective is to build a classification model that predicts credit default (0 = No Default, 1 = Default).



**Your Mission:** Can you discover the 'golden feature' - the most powerful predictor hidden in this anonymized data?



Your work will be evaluated based on the completion of the following tasks:


# Part 1: Read Data



**Tasks:**



1. Read the dataset `Q3_data.csv` using `read_csv()`

2. Inspect the first few rows using `head()`

3. Display dataset information using `info()`

4. Show statistical description using `describe()`

5. Plot the target distribution (target column)"


In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# The 'path' variable is defined in the download cell
df = pd.read_csv(os.path.join(path, 'Q3_data.csv'))


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df['Target'].value_counts().plot(kind='bar')
plt.title('Target Distribution (0: No Default, 1: Default)')
plt.xlabel('Target')
plt.ylabel('Count')
plt.show()


# Part 2: Data Cleaning



**Tasks:**



Inspect and fix the following when needed:



1. **Handle missing values appropriately**

2. **Check and remove duplicates** if any exist

3. **Encode categorical variables** if needed

4. **Apply feature scaling** to numerical features (Use StandardScaler)

5. **Check for target imbalance and state if it is imbalanced or not**

In [ ]:
print(df.isnull().sum())
df = df.fillna(df.median())


In [ ]:
print(f'Duplicates: {df.duplicated().sum()}')
df = df.drop_duplicates()


In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    le = LabelEncoder()
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    print(f'Encoded columns: {list(categorical_cols)}')


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_cols = df.drop(columns=['Target']).columns
df[X_cols] = scaler.fit_transform(df[X_cols])


In [ ]:
df['Target'].value_counts()
# there is imbalance

# Part 3: Modeling



**Tasks:**



1. Split the dataset into features (X) and target (y)

2. Use the correct split: **KFold** OR **StratifiedKFold**

3. Train a **CatBoostClassifier** model

4. Evaluate using the appropriate metric only (Accuracy vs. F1 Score).

5. Print the averaged score across all folds

In [ ]:
X = df.drop(columns=['Target'])
y = df['Target']


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(n_estimators=100, random_state=42)

f1 = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    f1.append(f1_score(y_val, preds))

print(f'F1: {np.mean(f1):.4f}')


# Part 4: Find the Golden Feature!



**Tasks:**



1. Plot feature importance from your trained model

2. Identify and print the name of the most important feature (the 'golden feature')

3. Plot the distribution of predictions

In [ ]:
importances = model.feature_importances_
feat_importances = pd.Series(importances, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_importances[:10].plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance')
plt.show()


In [ ]:
golden_feature = feat_importances.index[0]
print(f'The Golden Feature is: {golden_feature}')
print(f'Importance: {feat_importances.iloc[0]:.4f}')


# Part 5: Bonus - Retrain with Golden Feature Only



**Task:**



Now that you've found the golden feature, let's see how powerful it really is!



Retrain your CatBoostClassifier using **ONLY the golden feature** and compare its performance to the full model:



1. Create new X with only the golden feature

2. Run the same KFold loop with this single feature

3. Print and compare the accuracy with the full model

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(n_estimators=100, random_state=42)

# Retrain with only the golden feature
X_golden = X[[golden_feature]]

golden_accuracies = []
for train_idx, val_idx in skf.split(X_golden, y):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    golden_accuracies.append(f1_score(y_val, preds))

print(f'Accuracy with ONLY {golden_feature}: {np.mean(golden_accuracies):.4f}')
print(f'Full Model Accuracy: {np.mean(f1):.4f}')
print(f'Difference: {np.mean(f1) - np.mean(golden_accuracies):.4f}')
